# 🗂 Bases de datos vectoriales

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 4 — 45 minutos**

---

## Objetivo

Almacenar embeddings de texto en ChromaDB y realizar búsqueda semántica — el corazón de todo sistema RAG.

## Al terminar este bloque vas a poder:

1. Crear colecciones en ChromaDB y agregar documentos con metadata.
2. Buscar por similitud semántica y compararlo con búsqueda por palabras exactas.
3. Usar un modelo multilingüe para mejorar resultados en español argentino.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Base de datos vectorial** | Motor especializado en guardar y consultar embeddings con baja latencia. |
| **ChromaDB** | Base vectorial open-source embebida en Python — sin servidor externo. |
| **Similitud coseno** | Mide la cercanía entre dos vectores: 1 = idénticos, 0 = sin relación. |
| **Colección** | Grupo de documentos relacionados en ChromaDB, como una tabla en SQL. |
| **CRUD vectorial** | `add` / `get` / `update` / `delete` / `query` — las operaciones básicas. |

In [ ]:
!uv pip install -qq chromadb sentence-transformers

In [ ]:
# Verificamos que PyTorch está instalado (necesario para sentence-transformers)
import torch
print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

## ¿Para qué sirve una base de datos vectorial?

### Analogía

Una base SQL busca por coincidencia exacta: si preguntás por 'asado', solo te devuelve registros que tengan la palabra 'asado'. Una base vectorial busca por significado: 'carne a las brasas', 'parrilla', 'bife' son conceptualmente cercanos a 'asado' aunque no compartan ninguna letra.

### Dónde vive esto en el mundo real

ChromaDB, Pinecone y Weaviate son la infraestructura invisible de los asistentes de documentos de Notion, Slack y GitHub Copilot. Cada vez que un chatbot empresarial responde con información de un manual específico, hay una base vectorial detrás recuperando los fragmentos más relevantes. En el próximo notebook vas a conectar ChromaDB con Gemini para construir exactamente eso.

### ✎ Para pensar

- ¿Por qué ChromaDB usa similitud coseno en lugar de distancia euclidiana para comparar embeddings?
- Si agregás un documento en inglés a una colección con textos en español, ¿el modelo base (`all-MiniLM`) lo va a comparar bien con consultas en español?

In [ ]:
import chromadb

# Creamos un cliente de ChromaDB en memoria
# Nota: Los datos se pierden al cerrar el notebook
# Para persistencia, usar: chromadb.PersistentClient(path="./chroma_db")
client = chromadb.Client()

print("ChromaDB inicializado correctamente")

## Demo: base de reviews de restaurantes porteños

Vamos a construir un recomendador semántico con reviews reales. El objetivo es que una consulta como 'algo barato para morfar' encuentre restaurantes relevantes aunque no usen esas palabras exactas.

In [ ]:
collection = client.get_or_create_collection(
    name="reviews_restaurantes"
)
print("Colección 'reviews_restaurantes' lista")

In [ ]:
# Reviews iniciales - variedad de restaurantes en CABA
reviews_iniciales = [
    "Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto increíbles. Precio razonable, como 8 lucas por persona. Ambiente tranquilo, perfecto para ir en pareja.",

    "La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.",

    "Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.",

    "Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las papas fritas caseras. Atención familiar, te hacen sentir como en tu casa. Precios normales.",

    "Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el mejor que probé pero para el precio está más que bien. El combo para dos es suficiente."
]

# Metadatos: información adicional sobre cada review
metadatos_iniciales = [
    {"barrio": "San Telmo", "tipo": "italiana", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "pizzeria", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "gourmet", "precio": "alto"},
    {"barrio": "Villa Urquiza", "tipo": "parrilla", "precio": "medio"},
    {"barrio": "delivery", "tipo": "sushi", "precio": "medio"}
]

# IDs únicos para cada documento
ids_iniciales = ["review1", "review2", "review3", "review4", "review5"]

### Estructura de una colección

Cada `add()` necesita tres listas de igual longitud:

| Parámetro | Contenido |
|---|---|
| `documents` | Los textos en sí |
| `metadatas` | Diccionarios con datos extra (barrio, tipo, precio) |
| `ids` | Identificadores únicos para cada documento |

In [ ]:
# Agregamos las reviews a la colección
collection.add(
    documents=reviews_iniciales,
    metadatas=metadatos_iniciales,
    ids=ids_iniciales
)

print(f"Se agregaron {len(reviews_iniciales)} reviews a la base de datos")

In [ ]:
# Verificamos cuántos documentos tenemos
collection.count()

In [ ]:
# Obtenemos todos los documentos para verificar
todos = collection.get()
print("Documentos en la colección:")
for i, doc in enumerate(todos['documents'], 1):
    print(f"\n{i}. {doc[:80]}...")

## Búsqueda semántica vs búsqueda por palabras

Vas a comparar los dos tipos de búsqueda con los mismos datos para ver concretamente la diferencia.

In [ ]:
# Búsqueda 1: Quiero comer algo italiano
consulta = "Busco un lugar para comer buena comida italiana, tipo ravioles o ñoquis"

resultados = collection.query(
    query_texts=[consulta],
    n_results=3  # Los 3 más similares
)

print(f"CONSULTA: {consulta}")
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)
for i, (doc, metadata) in enumerate(zip(resultados['documents'][0], resultados['metadatas'][0]), 1):
    print(f"\n{i}. {doc}")
    print(f"   Barrio: {metadata['barrio']}, Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

In [ ]:
# Búsqueda 2: Lugar económico para carne
consulta2 = "Dónde puedo ir a comer buen asado sin gastar mucha plata?"

resultados2 = collection.query(
    query_texts=[consulta2],
    n_results=3
)

print(f"CONSULTA: {consulta2}")
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)
for i, (doc, metadata) in enumerate(zip(resultados2['documents'][0], resultados2['metadatas'][0]), 1):
    print(f"\n{i}. {doc}")
    print(f"   Barrio: {metadata['barrio']}, Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

In [ ]:
# Búsqueda tradicional: buscar la palabra "asado" exacta
print("BÚSQUEDA TRADICIONAL (palabra exacta 'asado'):")
busqueda_tradicional = collection.get(
    where_document={"$contains": "asado"}
)

if len(busqueda_tradicional['documents']) > 0:
    for doc in busqueda_tradicional['documents']:
        print(f"- {doc}")
else:
    print("No se encontraron resultados con la palabra 'asado'")

print("\n" + "="*80)
print("\nBÚSQUEDA SEMÁNTICA (por significado):")
print("Encontró la parrilla aunque no use la palabra 'asado' exacta")

### ✎ Para pensar

- La búsqueda encontró 'ñoquis' ante la consulta 'comida italiana'. ¿Qué tiene que saber el modelo de embeddings para hacer esa conexión?
- Probá la consulta 'lugar barato para morfar' con el modelo base y el multilingüe. ¿Cuál da mejor resultado? ¿Por qué?

In [ ]:
# Nuevas reviews que llegan
nuevas_reviews = [
    "Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de semana. Se llena bastante después de las 11. Tiene terraza.",

    "Cafetería de especialidad, café re rico, tienen opciones veganas. Ambiente tranquilo para trabajar con la compu. WiFi gratis y enchufes.",

    "Bodegón español tradicional. Las croquetas y la tortilla de papa son de otro planeta. Vino de la casa riquísimo. Porteño viejo, 100 años de historia.",

    "Hamburguesería gourmet. Las papas con cheddar y bacon son adictivas. Burgers de 200gr, jugosas. Podes armar la tuya. Delivery hasta las 2am.",

    "Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu nona. Milanesas gigantes. Super económico, 6 lucas comes re bien."
]

nuevos_metadatos = [
    {"barrio": "Palermo", "tipo": "bar", "precio": "medio-alto"},
    {"barrio": "Colegiales", "tipo": "cafeteria", "precio": "medio"},
    {"barrio": "Montserrat", "tipo": "española", "precio": "medio"},
    {"barrio": "Belgrano", "tipo": "hamburguesas", "precio": "medio"},
    {"barrio": "Boedo", "tipo": "casera", "precio": "bajo"}
]

In [ ]:
# Función helper para agregar reviews de forma organizada
def agregar_reviews(collection, reviews, metadatos, prefijo_id="review"):
    """
    Agrega nuevas reviews a una colección existente.

    Parámetros:
    -----------
    collection : chromadb.Collection
        La colección donde agregar los documentos
    reviews : list
        Lista de textos de reviews
    metadatos : list
        Lista de diccionarios con metadatos
    prefijo_id : str
        Prefijo para generar IDs únicos
    """
    # Obtenemos cuántos documentos ya hay para generar IDs únicos
    count_actual = collection.count()

    # Generamos IDs únicos
    nuevos_ids = [f"{prefijo_id}{count_actual + i + 1}" for i in range(len(reviews))]

    # Agregamos a la colección
    collection.add(
        documents=reviews,
        metadatas=metadatos,
        ids=nuevos_ids
    )

    print(f"Se agregaron {len(reviews)} nuevas reviews")
    print(f"Total de documentos en la colección: {collection.count()}")
    return nuevos_ids

In [ ]:
# Agregamos las nuevas reviews
ids_nuevos = agregar_reviews(collection, nuevas_reviews, nuevos_metadatos)

## Mejorando con embeddings multilingüe

El modelo por defecto (`all-MiniLM-L6-v2`) fue entrenado mayormente en inglés. Para modismos como 're copado', 'posta', 'morfar', necesitamos `multilingual-e5-large`.

**Trade-off:** más preciso para español, pero más lento y ocupa más memoria.

In [ ]:
collection_mejorada = client.get_or_create_collection(
    name="reviews_multilenguaje",
    embedding_function=modelo_multilenguaje
)
print("✓ Colección 'reviews_multilenguaje' lista con modelo multilenguaje")
print("  Esta colección entiende modismos rioplatenses mucho mejor que la base")

In [ ]:
# Consulta con modismos argentinos
consulta_argentina = "Busco un lugar re copado para morfar algo barato y llenadero"

print("="*80)
print(f"CONSULTA: {consulta_argentina}")
print("="*80)

# Búsqueda con modelo base
print("\n1. MODELO BASE (all-MiniLM-L6-v2):")
print("-"*80)
resultado_base = collection.query(
    query_texts=[consulta_argentina],
    n_results=3
)
for i, doc in enumerate(resultado_base['documents'][0], 1):
    print(f"\n{i}. {doc[:100]}...")

# Búsqueda con modelo multilenguaje
print("\n" + "="*80)
print("\n2. MODELO MULTILENGUAJE (multilingual-e5-large):")
print("-"*80)
resultado_mejorado = collection_mejorada.query(
    query_texts=[consulta_argentina],
    n_results=3
)
for i, (doc, metadata) in enumerate(zip(resultado_mejorado['documents'][0], resultado_mejorado['metadatas'][0]), 1):
    print(f"\n{i}. {doc[:100]}...")
    print(f"   Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

### 🐛 Laboratorio de Rotura

El código de abajo *parece* correcto. Antes de ejecutarlo, predecí: ¿qué va a pasar si intentás crear una colección con un nombre que ya existe?

```python
colision = client.create_collection(name="reviews_restaurantes")
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Cómo deberías manejar esto en un sistema real que se reinicia?

No mires la explicación todavía.

In [ ]:
# ── Momento 1 — La colisión de nombres ──
try:
    colision = client.create_collection(name="reviews_restaurantes")
    print("Se creó... pero ¿cómo puede haber dos colecciones con el mismo nombre?")
except Exception as e:
    print(f"✗ Error: {e}")
    print()
    print("ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.")

# ── Momento 2 — La solución idiomática ──
# get_or_create_collection evita el error: si existe la devuelve, si no la crea
collection_segura = client.get_or_create_collection(name="reviews_restaurantes")
print(f"\n✓ get_or_create_collection: {collection_segura.count()} documentos ya cargados")
print("   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.")

### 🐛 Laboratorio de Rotura

El código de abajo *parece* correcto. Antes de ejecutarlo, predecí: ¿qué va a pasar si intentás crear una colección con un nombre que ya existe?

```python
colision = client.create_collection(name="reviews_restaurantes")
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Cómo deberías manejar esto en un sistema real que se reinicia?

No mires la explicación todavía.

In [ ]:
# ── Momento 1 — La colisión de nombres ──
try:
    colision = client.create_collection(name="reviews_restaurantes")
    print("Se creó... pero ¿cómo puede haber dos colecciones con el mismo nombre?")
except Exception as e:
    print(f"✗ Error: {e}")
    print()
    print("ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.")

# ── Momento 2 — La solución idiomática ──
# get_or_create_collection evita el error: si existe la devuelve, si no la crea
collection_segura = client.get_or_create_collection(name="reviews_restaurantes")
print(f"\n✓ get_or_create_collection: {collection_segura.count()} documentos ya cargados")
print("   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.")

In [ ]:
# --- Espacio de practica ---
#
# Proba estas busquedas y compara los resultados:
#   1. 'Un lugar romantico para ir con mi pareja'
#   2. 'Algo para comer rapido al mediodia'
#   3. 'Comida como la que hace la abuela'
#
# Para cada una:
#   - Ejecuta con el modelo base (collection)
#   - Ejecuta con el modelo multilingue (collection_mejorada)
#   - Registra si los resultados tienen sentido semantico
#
MI_CONSULTA = 'Un lugar romantico para ir con mi pareja'

mis_resultados = collection_mejorada.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
for doc, meta in zip(mis_resultados['documents'][0], mis_resultados['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: explicale en treinta segundos, en voz alta o por escrito, la diferencia entre búsqueda semántica y búsqueda por palabras exactas a alguien que nunca escuchó hablar de vectores. Usá el ejemplo del asado si te ayuda. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

## Vista previa: de la búsqueda a RAG

Lo que hiciste hasta acá es la mitad de RAG: la parte de **Retrieval** (recuperación).

| Paso | Qué hace | Estado |
|---|---|---|
| Guardar documentos con embeddings | ChromaDB.add() | ✅ hecho |
| Buscar los más relevantes | ChromaDB.query() | ✅ hecho |
| Pasarlos como contexto al LLM | prompt con contexto | próximo notebook |
| Generar respuesta fundamentada | Gemini | próximo notebook |

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **ChromaDB** | Base vectorial embebida, sin servidor externo |
| **Colección** | Grupo de documentos con embeddings, metadatos e IDs |
| **Búsqueda semántica** | Encuentra por significado, no por palabras exactas |
| **Multilingual-e5** | Modelo de embeddings optimizado para español regional |
| **CRUD vectorial** | add / get / update / delete / query |

**Próximo bloque →** RAG completo: vas a conectar ChromaDB con Gemini y construir un sistema que responde preguntas sobre tus propios documentos.